# Load Data

In [ ]:
import jax
import jax.numpy as jnp
import grain.python as grain
import tiktoken
from pathlib import Path

In [ ]:
from src.utils import load_stories_from_file

## Sanity check content of the data set

In [ ]:
file_path = Path("../data/TinyStories-1000.txt")
delimeter: str = "<|endoftext|>"

with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
    data = f.read()
    stories = data.split(delimeter)

    print("first 300 characters:\n")
    print(stories[0][:300], "...")

    print(f"Total count of stories: {len(stories)}")

## Tokenizer

In [ ]:
tokenizer: tiktoken.Encoding = tiktoken.get_encoding("gpt2")

print(f"Vocabulary size of tokenizer: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

## Dataset Class

In [ ]:
class StoryDataset:
    
    def __init__(self, stories: list[str], maxlen: int, tokenizer: tiktoken.Encoding) -> None:
        self.stories: list[str] = stories
        self.maxlen: int = maxlen
        self.tokenizer: tiktoken.Encoding = tokenizer

        # constructs the token id for the delimeter so it can locate it in the future
        self.delimeter_token: int = tokenizer.encode('<|endoftext|>', \
                        allowed_special={'<|endoftext|>'})[0]

    def __len__(self) -> int:
        return len(self.stories)

    def __getitem__(self, idx: int) -> list[int]:
        story: str = self.stories[idx]

        # encode the entire story at that index (a string) into a list of token IDs using the tokenizer
        tokens: list[int] = self.tokenizer.encode(story, 
                                       allowed_special={'<|endoftext|>'})

        # truncate when the resulting tokens exceed the maximum set in the initializer
        if len(tokens) > self.maxlen:
            tokens: list[int] = tokens[:self.maxlen]

        # padding: add zeros to make sure all returned lists are the same size to input them all to the same tensors
        tokens.extend([0] * (self.maxlen - len(tokens)))
        
        return tokens

## Data Iterator (including shuffling with grain)

In [ ]:
shuffled_sampler = grain.IndexSampler(
    num_records=10,
    shuffle=True,
    seed=42,
    shard_options=grain.NoSharding(),    
    num_epochs=1
)

def print_sampler_example(sampler, name):
    print(f"\n{name}")
    for i, idx in enumerate(sampler):
        print(f"Record {i}: {idx}")

print_sampler_example(shuffled_sampler, "Shuffled sampler")

## Batch sequences into fixed-shape arrays

In [ ]:
batch_op_keep = grain.Batch(
    batch_size=32,
    drop_remainder=False # don't drop the remainder if if there is one
)

## Build a data loader

In [ ]:
def create_dataloader(
    stories: list[str],
    tokenizer: tiktoken.Encoding,
    maxlen: int,
    batch_size: int,
    shuffle: bool = False, # defaults to no shuffle
    num_epochs: int = 1,
    seed: int = 42,
    worker_count: int = 0
 ) -> (grain.DataLoader, int):
    
    dataset: StoryDataset = StoryDataset(stories, maxlen, tokenizer)
    estimated_batches: int = len(dataset) // batch_size

    # creates a new sampler
    sampler = grain.IndexSampler(
        num_records=len(dataset), # 1,000 stories for this dataset
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )

    # create new data loader
    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,

        # provide the operations that we want to perform on the dataset
        # in this case, the only operation being performed is batching
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        
        worker_count=worker_count
    )
    
    return dataloader, estimated_batches

## Execute the code

In [ ]:
# load the stories
stories = load_stories_from_file(
    "../data/TinyStories-1000.txt", 
    max_stories=100
)

# sanity check the first story
print(f"\n\n{stories[0]}")

In [ ]:
# create the data loader
dataloader, batches_per_epoch = create_dataloader(
    stories=stories,
    tokenizer=tokenizer,
    maxlen=128,
    batch_size=32, # small number of batches to run on a local machine
    shuffle=False,
    num_epochs=1,
    seed=42,
    worker_count=0  # Single process for experimentation
)

print(f"\nDataLoader created successfully:")
print(f"Will produce {batches_per_epoch} batches per epoch")

## Fetch data

In [ ]:
# create iterator and call next on it
my_iterator = iter(dataloader)
next(my_iterator)